# Vase Selection — 1D CNN Classifier

## Goal
Predict which vase a participant will select, given the shape of the vase as a profile curve.

Each vase is defined by a sequence of `(x, y)` coordinate pairs that trace its silhouette. Participants choose between two vases per trial across 5 rounds of 5 generations. The model learns which profile shapes tend to be selected across participants, then generalises to new participants it has never seen.

## Pipeline Overview
| Step | What happens |
|---|---|
| **1. Load data** | Read the raw parquet — one row per `(x, y)` sample point |
| **2. Build trajectories** | Group rows into per-participant, per-vase profile curves |
| **3. Preprocess** | Pad to uniform length, normalise each channel |
| **4. Split** | Divide by `participant_id` to avoid data leakage |
| **5. Model** | 1D CNN reads the profile curve and outputs a selection probability |
| **6. Train** | Binary cross-entropy loss, Adam optimiser, learning-rate schedule |

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [2]:
dataset = pd.read_parquet("../file_processing/processed/xy_data.parquet")
print("Dataset shape:", dataset.shape)
dataset.head()

Dataset shape: (20175000, 21)


,generation_key,vase,x,y,is_selected_vase,is_parent_vase,selection_elapsed_time_s,decision_time_ms,adjusted_selection_elapsed_time_s,adjusted_decision_time_ms,...,rt_calibration_status,rt_calibration_median_ms,rt_calibration_trials_n,participant_id,session_id,play_counter,metadata_schema_version,session_started_at,session_duration_s,completion_status
0,round_0_gen_1,vase_1,0.000650,1.977799,True,False,7.435,7435,7.427,7427.0,...,completed,690.0,8,01553c98-79fa-475e-a939-772f9f3d5e18,89013e10-fd8e-4c68-a2fb-f3917c35a895,1,1,1.778869e+09,119.029303,completed
1,round_0_gen_1,vase_1,0.046236,1.977766,True,False,7.435,7435,7.427,7427.0,...,completed,690.0,8,01553c98-79fa-475e-a939-772f9f3d5e18,89013e10-fd8e-4c68-a2fb-f3917c35a895,1,1,1.778869e+09,119.029303,completed
2,round_0_gen_1,vase_1,0.091811,1.977784,True,False,7.435,7435,7.427,7427.0,...,completed,690.0,8,01553c98-79fa-475e-a939-772f9f3d5e18,89013e10-fd8e-4c68-a2fb-f3917c35a895,1,1,1.778869e+09,119.029303,completed
3,round_0_gen_1,vase_1,0.137535,1.977849,True,False,7.435,7435,7.427,7427.0,...,completed,690.0,8,01553c98-79fa-475e-a939-772f9f3d5e18,89013e10-fd8e-4c68-a2fb-f3917c35a895,1,1,1.778869e+09,119.029303,completed
4,round_0_gen_1,vase_1,0.183402,1.978020,True,False,7.435,7435,7.427,7427.0,...,completed,690.0,8,01553c98-79fa-475e-a939-772f9f3d5e18,89013e10-fd8e-4c68-a2fb-f3917c35a895,1,1,1.778869e+09,119.029303,completed


## Step 2 — Build Trajectories

### What this does
The raw dataset has one row per sampled point on a vase profile. We group rows by `(participant_id, generation_key, vase)` to reconstruct each vase's full profile curve as seen by each participant.

Each group becomes one **trajectory** — a 2D array of shape `(n_points, 2)` where the columns are `x` (horizontal position along the vase) and `y` (height / radial distance).

### Why include `participant_id` in the groupby
The same vase shape appears once per participant who viewed it. Including `participant_id` means:
- Each trajectory is one sample in the dataset (one participant × one vase × one generation)
- Labels can differ across participants for the same vase shape (not everyone agrees)
- We can split the data cleanly by participant later (see Step 4)

### Outputs
| Variable | Description |
|---|---|
| `sequences` | List of `(n_points, 2)` numpy arrays — one per trajectory |
| `labels` | List of ints — `1` if this participant selected this vase, `0` otherwise |
| `participant_ids` | Parallel array of participant ID strings, used for the train/test split |

In [4]:
nn_dataset = dataset[["participant_id", "generation_key", "vase", "x", "y", "is_selected_vase"]].copy()
nn_dataset = nn_dataset.sort_values(["participant_id", "generation_key", "vase", "x"])

sequences, labels, participant_ids = [], [], []
for (pid, gen_key, vase), group in nn_dataset.groupby(
    ["participant_id", "generation_key", "vase"], sort=False
):
    sequences.append(group[["x", "y"]].values)  # (seq_len, 2)
    labels.append(int(group["is_selected_vase"].iloc[0]))
    participant_ids.append(pid)

participant_ids = np.array(participant_ids)
lengths = [len(s) for s in sequences]
print(f"Trajectories: {len(sequences)}")
print(f"Unique participants: {len(np.unique(participant_ids))}")
print(f"Sequence length — min: {min(lengths)}, max: {max(lengths)}, median: {int(np.median(lengths))}")
print(f"Selected: {sum(labels)} / {len(labels)} ({100*sum(labels)/len(labels):.1f}%)")

Trajectories: 80700
Unique participants: 1614
Sequence length — min: 250, max: 250, median: 250
Selected: 40350 / 80700 (50.0%)


## Step 3 — Preprocess: Pad & Normalise

### Padding (`SEQ_LEN`, `pad_sequence`)
A 1D CNN requires all inputs in a batch to be the same length. If sequences vary in length we pad the shorter ones with zeros at the end.

- **`SEQ_LEN = max(lengths)`** — pads every sequence to the length of the longest one. This is the safest default: no information is lost.
- **To truncate instead**: replace with a fixed integer, e.g. `SEQ_LEN = 500`. Useful if a few very long sequences are skewing memory usage. Sequences longer than `SEQ_LEN` will be cut at the end.

### Transposing to `(n_vases, 2, seq_len)`
PyTorch's `Conv1d` expects input shape `(batch, channels, length)`.
- **Channels = 2**: the `x` coordinate and the `y` coordinate are treated as two separate signal channels, just like R/G/B are three channels in an image CNN.
- After stacking we have `(n_vases, seq_len, 2)`, so `.transpose(0, 2, 1)` reorders to `(n_vases, 2, seq_len)`.

### Z-score Normalisation
Each channel is standardised independently: `(value − mean) / std`. This ensures:
- Neither coordinate dominates due to scale differences
- Gradient flow during training is well-conditioned

The normalisation is computed across the whole dataset (all vases, all points). If you add new data at inference time, you should save the training mean/std and reuse them rather than recomputing.

In [5]:
SEQ_LEN = max(lengths)  # pad to longest trajectory; swap for a fixed int to truncate

def pad_sequence(seq, length):
    if len(seq) >= length:
        return seq[:length]
    return np.vstack([seq, np.zeros((length - len(seq), 2))])

# Stack to (n_vases, 2, seq_len) — Conv1d expects (batch, channels, length)
X = np.stack([pad_sequence(s, SEQ_LEN) for s in sequences]).transpose(0, 2, 1).astype(np.float32)

# Per-channel z-score normalisation (ignoring padded zeros)
for c in range(2):
    vals = X[:, c, :]
    X[:, c, :] = (vals - vals.mean()) / (vals.std() + 1e-8)

y = np.array(labels, dtype=np.float32)
print(f"X: {X.shape}  y: {y.shape}")

X: (80700, 2, 250)  y: (80700,)


## Step 4 — Dataset & Participant-Based Split

### Why split by participant, not randomly
If we split rows (or trajectories) randomly, the same participant could appear in both train and test. The model could then memorise participant-specific preferences rather than learning generalisable shape features. Splitting by `participant_id` guarantees the test set contains only participants the model has never seen — this is a much more honest estimate of real-world performance.

### How the split works
1. `train_test_split` is called on the list of **unique participant IDs** (not on trajectories directly)
2. `np.isin(participant_ids, train_pids)` produces a boolean mask aligned with the `sequences`/`labels` arrays
3. The masks index into `X` and `y` to create the train and test subsets

### Parameters
| Parameter | Default | Effect |
|---|---|---|
| `test_size=0.2` | 20% of participants → test | Increase for a larger test set; decrease if you have few participants |
| `random_state=42` | Fixed seed | Change to get a different split; use cross-validation across multiple seeds for a more robust estimate |
| `batch_size=64` | 64 trajectories per batch | Larger batches → faster training but more memory; smaller batches → noisier gradients but can regularise |

In [6]:
class VaseDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Split on participant IDs so no participant appears in both sets
unique_pids = np.unique(participant_ids)
train_pids, test_pids = train_test_split(unique_pids, test_size=0.2, random_state=42)

train_mask = np.isin(participant_ids, train_pids)
test_mask  = np.isin(participant_ids, test_pids)

train_loader = DataLoader(VaseDataset(X[train_mask], y[train_mask]), batch_size=64, shuffle=True)
test_loader  = DataLoader(VaseDataset(X[test_mask],  y[test_mask]),  batch_size=64)

print(f"Train: {train_mask.sum()} trajectories from {len(train_pids)} participants")
print(f"Test:  {test_mask.sum()} trajectories from {len(test_pids)} participants")

Train: 64550 trajectories from 1291 participants
Test:  16150 trajectories from 323 participants


## Step 5 — Model Architecture: `VaseCNN`

The model reads a vase profile curve (two channels: `x` and `y`, over `seq_len` points) and outputs a single logit — a raw score that is converted to a probability by a sigmoid function.

```
Input:  (batch, 2, seq_len)
         ↓
Conv1d(2→32, kernel=7)  + ReLU  → (batch, 32, seq_len)
MaxPool1d(2)                     → (batch, 32, seq_len/2)
         ↓
Conv1d(32→64, kernel=5) + ReLU  → (batch, 64, seq_len/2)
MaxPool1d(2)                     → (batch, 64, seq_len/4)
         ↓
Conv1d(64→128, kernel=3) + ReLU → (batch, 128, seq_len/4)
AdaptiveAvgPool1d(1)             → (batch, 128, 1)   ← global average
         ↓
Flatten → Linear(128→64) + ReLU + Dropout(0.3) → Linear(64→1)
Output: (batch,)  ← raw logit per vase
```

### Convolutional layers — what each parameter does

| Parameter | What it controls |
|---|---|
| **`in_channels` / `out_channels`** | How many feature maps the layer reads and writes. More channels = more capacity to detect different curve patterns. Doubling channels doubles memory and compute. |
| **`kernel_size`** | How many consecutive points the filter looks at simultaneously. Larger kernels detect broader shape features (e.g. overall curvature); smaller kernels detect local detail (e.g. sharp inflections). |
| **`padding`** | Set to `kernel_size // 2` (e.g. `padding=3` for `kernel_size=7`) to keep the output the same length as the input. Without padding the sequence shrinks with each layer. |

### Pooling layers

- **`MaxPool1d(2)`**: halves the sequence length by taking the maximum in each non-overlapping window of 2. Reduces computation and forces the network to find the most important features rather than memorising position.
- **`AdaptiveAvgPool1d(1)`**: averages the entire remaining sequence into a single value per channel, producing a fixed-size `(batch, 128)` vector regardless of the original `seq_len`. This is what allows the network to handle variable-length inputs.

### Fully-connected head

- **`Linear(128→64)`**: compresses the CNN's 128-dimensional summary into 64 dimensions.
- **`Dropout(0.3)`**: randomly zeros 30% of activations during training. This is regularisation — it forces the network not to rely too heavily on any single feature, which reduces overfitting. Set to `0` to disable.
- **`Linear(64→1)`**: produces a single raw score (logit). Positive → predicts selected; negative → predicts not selected.

---

### How to tune the architecture

**More capacity (if underfitting — train accuracy is low):**
- Add a fourth Conv1d block, e.g. `Conv1d(128, 256, kernel_size=3, padding=1)`
- Increase channel counts: `32→64→128→256` instead of `32→64→128`
- Increase the hidden layer size: `Linear(128→128)` instead of `Linear(128→64)`

**Less capacity (if overfitting — train accuracy is high but test accuracy is low):**
- Increase `Dropout` from `0.3` to `0.5`
- Add `nn.BatchNorm1d(channels)` after each Conv1d (also speeds up training)
- Reduce channel counts or remove the third conv block

**Larger receptive field (if the model needs to see more of the curve at once):**
- Increase `kernel_size` in early layers, e.g. `kernel_size=15`
- Add dilated convolutions: `nn.Conv1d(..., dilation=2)` — looks at every other point, doubling the effective window without extra parameters

In [7]:
class VaseCNN(nn.Module):
    def __init__(self, in_channels=2):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=7, padding=3), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),           nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),          nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),  # → (batch, 128, 1) regardless of seq_len
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.head(self.conv(x)).squeeze(1)

model = VaseCNN().to(device)
print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")

VaseCNN(
  (conv): Sequential(
    (0): Conv1d(2, 32, kernel_size=(7,), stride=(1,), padding=(3,))
    (1): ReLU()
    (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (4): ReLU()
    (5): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (7): ReLU()
    (8): AdaptiveAvgPool1d(output_size=1)
  )
  (head): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=128, out_features=64, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)

Parameters: 43,809


## Step 6 — Training

### Loss function: `BCEWithLogitsLoss`
Binary Cross-Entropy with Logits. It applies a sigmoid internally, which is numerically more stable than applying `sigmoid` first and then computing log loss separately. It outputs a value near 0 when the model's prediction matches the label, and a large positive value when it's wrong.

### Optimiser: Adam
Adam adapts the learning rate for each parameter individually based on recent gradient history. It requires very little tuning and works well as a default.

| Parameter | Default | Effect |
|---|---|---|
| `lr=1e-3` | Learning rate | How large each weight update step is. If loss isn't decreasing, try `1e-4`. If training is very slow, try `3e-3`. |

### Learning-rate schedule: `ReduceLROnPlateau`
Watches the test loss. If it hasn't improved for `patience` epochs, it multiplies the learning rate by `factor`. This prevents stagnation late in training.

| Parameter | Default | Effect |
|---|---|---|
| `patience=3` | Epochs without improvement before reducing LR | Increase if your loss is noisy and you don't want premature reductions |
| `factor=0.5` | Multiply LR by this when triggered | `0.5` halves the LR each time; `0.1` reduces it more aggressively |

### `EPOCHS = 20`
One epoch = one full pass through the training data. 20 is a reasonable starting point. Watch for:
- **Overfitting**: test loss starts rising while train loss keeps falling → reduce epochs or add regularisation
- **Underfitting**: both losses are still falling at epoch 20 → increase epochs

---

## Adding Demographic Data to Improve Performance

Vase preferences may differ systematically by age, gender, cultural background, etc. Demographic features can be incorporated by concatenating them with the CNN's output vector before the final classifier.

### How to implement it

**1. Build a demographic feature array aligned with your trajectories:**
```python
# Example: age (normalised), gender (one-hot), etc.
# Shape: (n_trajectories, n_demo_features)
demo_features = ...
```

**2. Modify `VaseDataset` to carry demographic features:**
```python
class VaseDataset(Dataset):
    def __init__(self, X, y, demo):
        self.X    = torch.from_numpy(X)
        self.y    = torch.from_numpy(y)
        self.demo = torch.from_numpy(demo.astype(np.float32))

    def __getitem__(self, idx):
        return self.X[idx], self.demo[idx], self.y[idx]
```

**3. Modify `VaseCNN` to accept demographic features in the forward pass:**
```python
class VaseCNN(nn.Module):
    def __init__(self, in_channels=2, n_demo=5):
        super().__init__()
        self.conv = nn.Sequential(...)   # unchanged
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 + n_demo, 64),  # ← 128 from CNN + n_demo features
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1),
        )

    def forward(self, x, demo):
        cnn_out = self.conv(x).squeeze(-1)   # (batch, 128)
        combined = torch.cat([cnn_out, demo], dim=1)  # (batch, 128 + n_demo)
        return self.head(combined).squeeze(1)
```

### What demographic features are likely useful

| Feature | How to encode | Why it might help |
|---|---|---|
| Age | Single float, z-scored | Aesthetic preferences shift with age |
| Gender | One-hot (e.g. [1,0,0]) | Well-documented differences in shape preferences |
| Cultural background | One-hot or embedding | Cross-cultural aesthetics research shows systematic differences |
| Expertise / art training | Binary flag | Trained eyes may prefer different proportions |
| Decision time | Float (log-scaled) | Fast decisions may reflect gut preference; slow ones deliberation |

### Caveats
- Demographic features are participant-level: the same values repeat for every trajectory belonging to that participant. This is fine — the model still sees them paired with different vase shapes.
- If demographic data is missing for some participants, you can either drop those participants or impute (e.g. with the training-set mean).
- With small datasets, demographic features risk overfitting. Use `Dropout` on the combined vector and monitor the train/test gap carefully.

In [9]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

def evaluate(loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            total_loss += criterion(logits, y_batch).item() * len(y_batch)
            correct += ((torch.sigmoid(logits) > 0.5) == y_batch.bool()).sum().item()
            n += len(y_batch)
    return total_loss / n, correct / n

EPOCHS = 20
for epoch in range(1, EPOCHS + 1):
    model.train()
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        criterion(model(X_batch), y_batch).backward()
        optimizer.step()

    tr_loss, tr_acc = evaluate(train_loader)
    te_loss, te_acc = evaluate(test_loader)
    scheduler.step(te_loss)
    print(f"Epoch {epoch:2d}/{EPOCHS}  train loss {tr_loss:.4f} acc {tr_acc:.3f}  |  test loss {te_loss:.4f} acc {te_acc:.3f}")

Epoch  1/20  train loss 0.6867 acc 0.546  |  test loss 0.6864 acc 0.547
Epoch  2/20  train loss 0.6866 acc 0.549  |  test loss 0.6866 acc 0.552
Epoch  3/20  train loss 0.6831 acc 0.558  |  test loss 0.6831 acc 0.558
Epoch  4/20  train loss 0.6821 acc 0.560  |  test loss 0.6819 acc 0.562
Epoch  5/20  train loss 0.6841 acc 0.561  |  test loss 0.6842 acc 0.560
Epoch  6/20  train loss 0.6808 acc 0.564  |  test loss 0.6811 acc 0.563
Epoch  7/20  train loss 0.6803 acc 0.565  |  test loss 0.6807 acc 0.565
Epoch  8/20  train loss 0.6800 acc 0.567  |  test loss 0.6808 acc 0.566
Epoch  9/20  train loss 0.6809 acc 0.564  |  test loss 0.6817 acc 0.563
Epoch 10/20  train loss 0.6793 acc 0.567  |  test loss 0.6806 acc 0.563
Epoch 11/20  train loss 0.6783 acc 0.568  |  test loss 0.6798 acc 0.566
Epoch 12/20  train loss 0.6778 acc 0.571  |  test loss 0.6797 acc 0.563
Epoch 13/20  train loss 0.6773 acc 0.570  |  test loss 0.6793 acc 0.562
Epoch 14/20  train loss 0.6773 acc 0.572  |  test loss 0.6797 ac